# 02-1. OpenCV 데이터 증강 학습

이 노트북은 `02_training_baseline.ipynb`에서 만든 기본 학습 흐름을 확장합니다.

핵심 목적은 단순히 점수를 높이는 것이 아니라, **왜 데이터를 늘렸는지, 어떻게 공정하게 비교했는지, 결과를 어떻게 해석할지**를 설명할 수 있게 만드는 것입니다.

## Step 0. [디벨롭] 이번 노트북을 따로 만드는 이유

`02`에서는 원본 train 데이터만 사용해서 baseline 모델을 학습했습니다.

이번 `02-1`에서는 OpenCV로 train 이미지만 변형해서 데이터 수를 늘린 뒤, 같은 모델 구조로 다시 학습합니다.

이렇게 따로 나누는 이유는 다음과 같습니다.

- baseline과 증강 모델을 비교하기 쉽습니다.
- 어떤 조건을 바꿨는지 명확하게 설명할 수 있습니다.
- 발표에서 `처음 모델 → 개선 시도 → 결과 비교` 흐름을 보여줄 수 있습니다.

## Step 1. [디벨롭] WHY: 왜 OpenCV 데이터 증강을 하는지 정리합니다

실제 강아지/고양이 사진은 항상 같은 조건에서 찍히지 않습니다. 조명이 밝거나 어두울 수 있고, 대비가 다를 수 있으며, 사진이 조금 흔들리거나 회전되어 있을 수도 있습니다.

모델이 원본 train 이미지만 보고 학습하면 이런 변화에 약할 수 있습니다. 그래서 OpenCV를 사용해 train 이미지를 여러 조건으로 변형하고, 모델이 더 다양한 상황을 경험하도록 합니다.

단, validation/test 이미지는 변형하지 않습니다. 그 이유는 모델 평가를 실제 원본 데이터 기준으로 해야 baseline과 공정하게 비교할 수 있기 때문입니다.

## Step 2. [디벨롭] HOW: 비교 조건을 고정합니다

이번 실험에서 바꾸는 것은 **train 데이터에 OpenCV 증강을 적용하는지 여부**입니다.

공정한 비교를 위해 다음 조건은 baseline과 최대한 동일하게 유지합니다.

- 같은 70/15/15 split CSV를 사용합니다.
- 같은 validation/test 데이터를 사용합니다.
- 같은 이미지 크기와 batch size를 사용합니다.
- 같은 SimpleCNN 구조를 사용합니다.
- 같은 loss function과 optimizer 계열을 사용합니다.
- EarlyStopping과 best model checkpoint 저장 흐름도 같은 방식으로 사용합니다.

## Step 3. [디벨롭] 필요한 패키지를 불러옵니다

`cv2`는 OpenCV를 파이썬에서 사용할 때 import하는 이름입니다.

OpenCV는 이미지 밝기, 대비, 회전, 흐림 처리 같은 변형을 직접 만들 때 사용합니다.

In [ ]:
from pathlib import Path
import time

import cv2  # [디벨롭] OpenCV로 이미지 밝기/대비/회전 같은 증강을 만들기 위해 추가했습니다.
import matplotlib.pyplot as plt
import numpy as np  # [디벨롭] OpenCV 증강 실험의 랜덤 조건을 고정하기 위해 사용합니다.
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm  # [디벨롭] 증강 이미지 생성 진행률을 보기 위해 추가했습니다.

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)  # [디벨롭] NumPy 기반 랜덤도 같은 결과가 나오도록 고정합니다.

## Step 4. [디벨롭] 프로젝트 경로와 저장 위치를 정합니다

증강 이미지는 `data/processed/augmented/opencv_train/`에 저장합니다.

모델 checkpoint와 그래프는 `outputs/` 아래에 저장합니다. 이 결과물들은 용량이 커질 수 있으므로 Git에는 올리지 않습니다.

In [ ]:
# 노트북을 프로젝트 루트에서 실행하든 notebooks 폴더 안에서 실행하든 경로가 맞도록 처리합니다.
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
AUGMENTED_IMAGE_DIR = PROCESSED_DIR / "augmented" / "opencv_train"  # [디벨롭] OpenCV로 새로 만든 train 이미지를 저장합니다.
FIGURE_DIR = PROJECT_DIR / "outputs" / "figures"
METRICS_DIR = PROJECT_DIR / "outputs" / "metrics"
CHECKPOINT_DIR = PROJECT_DIR / "outputs" / "checkpoints"

for directory in [AUGMENTED_IMAGE_DIR, FIGURE_DIR, METRICS_DIR, CHECKPOINT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SPLIT_CSV_PATH = PROCESSED_DIR / "pet_binary_split_70_15_15_breed_stratified.csv"
AUGMENTED_TRAIN_CSV_PATH = PROCESSED_DIR / "pet_binary_split_70_15_15_opencv_augmented_train.csv"  # [디벨롭] 원본 train + 증강 train 목록을 저장합니다.
CHECKPOINT_PATH = CHECKPOINT_DIR / "best_opencv_augmented_simplecnn.pt"  # [디벨롭] 증강 실험의 best model을 baseline과 따로 저장합니다.

SPLIT_CSV_PATH

## Step 5. 01에서 만든 split CSV를 불러옵니다

여기서는 데이터를 다시 나누지 않습니다.

이미 `01_data_split.ipynb`에서 breed 기준으로 train/validation/test를 나눴기 때문에, 같은 split을 재사용해야 baseline과 비교가 가능합니다.

In [ ]:
split_df = pd.read_csv(SPLIT_CSV_PATH)

train_df = split_df[split_df["split"] == "train"].reset_index(drop=True)
val_df = split_df[split_df["split"] == "validation"].reset_index(drop=True)
test_df = split_df[split_df["split"] == "test"].reset_index(drop=True)

pd.DataFrame({
    "split": ["train", "validation", "test"],
    "count": [len(train_df), len(val_df), len(test_df)],
})

## Step 6. [디벨롭] OpenCV 증강 함수를 만듭니다

여기서 함수는 `한 장의 이미지`를 받아서 `변형된 한 장의 이미지`를 돌려주는 역할을 합니다.

이번에는 너무 복잡한 증강보다 발표에서 설명하기 쉬운 변형을 먼저 사용합니다.

- 밝기 증가: 조명이 밝은 사진을 가정합니다.
- 대비 증가: 털과 배경의 차이가 더 강한 사진을 가정합니다.
- 흐림 처리: 사진이 약간 흔들린 상황을 가정합니다.
- 회전: 카메라가 조금 기울어진 상황을 가정합니다.
- 좌우 반전: 동물이 반대 방향을 보고 있는 상황을 가정합니다.

In [ ]:
# [디벨롭] 아래 함수들은 baseline에는 없고, OpenCV 증강 실험을 위해 새로 추가한 함수들입니다.
def brighten_image(image):
    # beta는 픽셀 값에 더해지는 밝기 값입니다. 값이 커질수록 이미지가 밝아집니다.
    return cv2.convertScaleAbs(image, alpha=1.0, beta=35)


def increase_contrast(image):
    # alpha는 대비를 조절하는 값입니다. 1보다 크면 밝고 어두운 차이가 더 커집니다.
    return cv2.convertScaleAbs(image, alpha=1.25, beta=0)


def blur_image(image):
    # GaussianBlur는 주변 픽셀을 섞어서 이미지를 부드럽게 만듭니다.
    # (5, 5)는 blur를 계산할 때 보는 주변 영역의 크기입니다.
    return cv2.GaussianBlur(image, (5, 5), 0)


def rotate_image(image):
    # 이미지를 8도만 회전합니다. 너무 크게 돌리면 원본 의미가 달라질 수 있습니다.
    height, width = image.shape[:2]
    center = (width / 2, height / 2)
    rotation_matrix = cv2.getRotationMatrix2D(center, angle=8, scale=1.0)
    return cv2.warpAffine(image, rotation_matrix, (width, height), borderMode=cv2.BORDER_REFLECT_101)


def flip_image(image):
    # 1은 좌우 반전을 의미합니다. 강아지/고양이 이진 분류에서는 좌우 방향이 정답을 바꾸지 않습니다.
    return cv2.flip(image, 1)


# [디벨롭] 어떤 증강을 적용할지 목록으로 관리합니다. 나중에 실험을 추가/삭제하기 쉽습니다.
AUGMENTATIONS = [
    {"name": "bright", "function": brighten_image},
    {"name": "contrast", "function": increase_contrast},
    {"name": "blur", "function": blur_image},
    {"name": "rotate", "function": rotate_image},
    {"name": "flip", "function": flip_image},
]

[augmentation["name"] for augmentation in AUGMENTATIONS]

## Step 7. [디벨롭] 증강 이미지 샘플을 먼저 확인합니다

이미지를 많이 만들기 전에, 한 장의 사진에 증강이 어떻게 적용되는지 눈으로 확인합니다.

이 과정은 나중에 오분류 분석에서도 중요합니다. 모델 결과만 보는 것이 아니라, 데이터가 실제로 어떤 모습인지 직접 확인해야 하기 때문입니다.

In [ ]:
# [디벨롭] 증강 이미지도 CSV에 상대 경로로 저장되므로, 실제 파일 경로로 바꿔주는 helper입니다.
def resolve_image_path(image_path):
    # CSV 안의 경로가 절대 경로면 그대로 쓰고, 상대 경로면 프로젝트 루트를 붙입니다.
    path = Path(image_path)
    if path.is_absolute():
        return path
    return PROJECT_DIR / path


# [디벨롭] 전체 이미지를 만들기 전에, 한 장의 샘플로 증강 결과를 먼저 눈으로 확인합니다.
sample_row = train_df.iloc[0]
sample_path = resolve_image_path(sample_row["image_path"])
sample_bgr = cv2.imread(str(sample_path))

if sample_bgr is None:
    raise FileNotFoundError(f"이미지를 읽을 수 없습니다: {sample_path}")

sample_images = [("original", sample_bgr)]
for augmentation in AUGMENTATIONS:
    sample_images.append((augmentation["name"], augmentation["function"](sample_bgr)))

plt.figure(figsize=(14, 6))
for index, (title, image_bgr) in enumerate(sample_images, start=1):
    # OpenCV는 BGR 순서를 사용하고, matplotlib은 RGB 순서를 사용하므로 색상 순서를 바꿉니다.
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    plt.subplot(2, 3, index)
    plt.imshow(image_rgb)
    plt.title(title)
    plt.axis("off")

plt.tight_layout()
plt.show()

## Step 8. [디벨롭] train 이미지만 증강해서 파일로 저장합니다

여기서 중요한 원칙은 **train만 증강한다**는 점입니다.

validation/test까지 증강하면 평가 기준 자체가 바뀌기 때문에 baseline과 비교하기 어려워집니다.

처음 빠르게 테스트하고 싶으면 `MAX_AUGMENT_ORIGINALS = 100`처럼 바꿀 수 있습니다. 최종 비교를 할 때는 `None`으로 두고 전체 train 이미지를 증강합니다.

In [ ]:
MAX_AUGMENT_ORIGINALS = None  # [디벨롭] 빠른 테스트는 100처럼 숫자를 넣고, 최종 실험은 None으로 둡니다.

if MAX_AUGMENT_ORIGINALS is None:
    train_source_df = train_df.copy()
else:
    train_source_df = train_df.sample(n=MAX_AUGMENT_ORIGINALS, random_state=RANDOM_SEED).reset_index(drop=True)

augmented_records = []  # [디벨롭] 새로 만든 증강 이미지의 metadata를 여기에 모읍니다.

for _, row in tqdm(train_source_df.iterrows(), total=len(train_source_df)):
    original_path = resolve_image_path(row["image_path"])
    original_bgr = cv2.imread(str(original_path))

    if original_bgr is None:
        print(f"건너뜁니다. 이미지를 읽을 수 없습니다: {original_path}")
        continue

    # [디벨롭] 원본 이미지 한 장에 여러 OpenCV 변형을 차례대로 적용합니다.
    for augmentation in AUGMENTATIONS:
        augmented_bgr = augmentation["function"](original_bgr)
        augmentation_name = augmentation["name"]
        output_dir = AUGMENTED_IMAGE_DIR / augmentation_name
        output_dir.mkdir(parents=True, exist_ok=True)

        output_path = output_dir / f"{original_path.stem}_{augmentation_name}.jpg"
        cv2.imwrite(str(output_path), augmented_bgr)  # [디벨롭] 증강 이미지를 실제 파일로 저장합니다.

        new_record = row.to_dict()
        new_record["image_path"] = str(output_path.relative_to(PROJECT_DIR))
        new_record["is_augmented"] = True  # [디벨롭] 나중에 원본/증강 이미지를 구분하기 위한 표시입니다.
        new_record["augmentation_type"] = augmentation_name
        new_record["original_image_path"] = row["image_path"]
        augmented_records.append(new_record)

len(augmented_records)

## Step 9. [디벨롭] 원본 train과 증강 train을 하나의 DataFrame으로 합칩니다

원본 train도 함께 사용합니다. 즉, 모델은 원본 이미지와 증강 이미지를 모두 보고 학습합니다.

`is_augmented`, `augmentation_type`, `original_image_path` 컬럼을 추가해두면 나중에 어떤 방식으로 만든 이미지인지 추적할 수 있습니다.

In [ ]:
# [디벨롭] 원본 train에도 metadata 컬럼을 맞춰서 증강 train과 하나로 합칠 수 있게 합니다.
original_train_metadata = train_df.copy()
original_train_metadata["is_augmented"] = False
original_train_metadata["augmentation_type"] = "original"
original_train_metadata["original_image_path"] = original_train_metadata["image_path"]

# [디벨롭] 02-1에서 실제 학습에 사용할 train DataFrame입니다. 원본 train과 증강 train이 함께 들어갑니다.
augmented_train_df = pd.concat(
    [original_train_metadata, pd.DataFrame(augmented_records)],
    ignore_index=True,
)

augmented_train_df.to_csv(AUGMENTED_TRAIN_CSV_PATH, index=False)  # [디벨롭] 증강된 train 목록을 재사용할 수 있게 CSV로 저장합니다.

pd.DataFrame({
    "dataset": ["original train", "augmented only", "train used for 02-1"],
    "count": [len(train_df), len(augmented_records), len(augmented_train_df)],
})

## Step 10. Dataset과 DataLoader를 만듭니다

여기서는 `02`와 같은 전처리를 사용합니다.

비교 실험에서는 전처리까지 갑자기 바꾸면 무엇 때문에 결과가 달라졌는지 설명하기 어려워집니다. 그래서 이번 실험에서는 train 데이터 증강 여부만 주요 차이로 둡니다.

In [ ]:
IMAGE_SIZE = 128
BATCH_SIZE = 32
NUM_WORKERS = 0

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
])

label_to_index = {"cat": 0, "dog": 1}


class PetBinaryDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image_path = resolve_image_path(row["image_path"])

        image = Image.open(image_path).convert("RGB")
        label = label_to_index[row["label"]]

        if self.transform is not None:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)


train_dataset = PetBinaryDataset(augmented_train_df, transform=transform)  # [디벨롭] baseline의 train_df 대신 증강 train_df를 사용합니다.
val_dataset = PetBinaryDataset(val_df, transform=transform)
test_dataset = PetBinaryDataset(test_df, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

len(train_loader), len(val_loader), len(test_loader)

## Step 11. baseline과 같은 SimpleCNN을 사용합니다

모델 구조를 바꾸지 않는 이유는 OpenCV 증강의 영향을 더 깔끔하게 보기 위해서입니다.

이번 실험에서 바뀌는 핵심은 모델이 아니라 train 데이터입니다.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 128),
            nn.ReLU(),
            nn.Linear(128, 2),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

## Step 12. [디벨롭] 개선 학습 루프로 증강 모델을 학습합니다

여기서는 `02`에서 만든 개선 학습 흐름을 다시 사용합니다.

- validation loss가 좋아질 때만 best model을 저장합니다.
- validation loss가 일정 횟수 동안 좋아지지 않으면 EarlyStopping으로 멈춥니다.
- 학습이 끝나면 best checkpoint를 다시 불러옵니다.

PyTorch에서는 Keras의 `EarlyStopping`, `ModelCheckpoint`처럼 자동 callback을 붙이는 대신, 조건문으로 직접 흐름을 작성합니다.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
def train_one_epoch(model, data_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in data_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()          # 1. 이전 batch의 gradient를 비웁니다.
        outputs = model(images)        # 2. 모델이 예측합니다. PyTorch에서는 이 줄에서 forward가 실행됩니다.
        loss = criterion(outputs, labels)  # 3. 정답과 비교해서 오차를 계산합니다.
        loss.backward()                # 4. 오차를 줄이는 방향을 계산합니다.
        optimizer.step()               # 5. 계산된 방향으로 가중치를 업데이트합니다.

        total_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


def evaluate(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

In [ ]:
model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

MAX_EPOCHS = 30
PATIENCE = 5
MIN_DELTA = 0.0

history = {
    "train_loss": [],
    "val_loss": [],
    "train_accuracy": [],
    "val_accuracy": [],
}

best_val_loss = float("inf")
epochs_without_improvement = 0

start_time = time.time()

for epoch in range(MAX_EPOCHS):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = evaluate(model, val_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_accuracy"].append(val_accuracy)

    print(
        f"epoch {epoch + 1}/{MAX_EPOCHS} | "
        f"train loss {train_loss:.4f}, train acc {train_accuracy:.4f} | "
        f"val loss {val_loss:.4f}, val acc {val_accuracy:.4f}"
    )

    if val_loss < best_val_loss - MIN_DELTA:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        torch.save(model.state_dict(), CHECKPOINT_PATH)  # [디벨롭] OpenCV 증강 실험의 best model을 별도 파일로 저장합니다.
        print(f"  -> best model 저장: {CHECKPOINT_PATH.name}")
    else:
        epochs_without_improvement += 1
        print(f"  -> 개선 없음: {epochs_without_improvement}/{PATIENCE}")

    if epochs_without_improvement >= PATIENCE:
        print("EarlyStopping 조건을 만족해서 학습을 멈춥니다.")
        break

elapsed_time = time.time() - start_time

model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.to(device)

best_val_loss, elapsed_time

## Step 13. [디벨롭] RESULT: loss curve와 학습 기록을 저장합니다

loss curve를 보면 train loss와 validation loss가 함께 내려가는지, 또는 train loss만 내려가고 validation loss가 다시 올라가는지 확인할 수 있습니다.

validation loss가 다시 올라가기 시작하면 모델이 train 데이터에만 너무 맞춰지는 overfitting 가능성을 의심할 수 있습니다.

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, history["train_loss"], marker="o", label="train loss")
plt.plot(epochs, history["val_loss"], marker="o", label="validation loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("OpenCV Augmented Train/Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

loss_curve_path = FIGURE_DIR / "opencv_augmented_loss_curve.png"  # [디벨롭] baseline loss curve와 구분되는 증강 실험 그래프입니다.
plt.savefig(loss_curve_path, dpi=150)
plt.show()

loss_curve_path

In [ ]:
# [디벨롭] 나중에 baseline과 비교할 수 있도록 증강 실험 결과를 CSV로 남깁니다.
training_summary = pd.DataFrame([
    {
        "experiment": "opencv_augmented_simplecnn",
        "original_train_count": len(train_df),
        "augmented_image_count": len(augmented_records),
        "train_count_used": len(augmented_train_df),
        "validation_count": len(val_df),
        "test_count": len(test_df),
        "epochs_ran": len(history["train_loss"]),
        "best_val_loss": best_val_loss,
        "best_val_accuracy": max(history["val_accuracy"]),
        "elapsed_time_seconds": elapsed_time,
        "checkpoint_path": str(CHECKPOINT_PATH.relative_to(PROJECT_DIR)),
        "loss_curve_path": str(loss_curve_path.relative_to(PROJECT_DIR)),
    }
])

summary_path = METRICS_DIR / "opencv_augmented_training_summary.csv"
training_summary.to_csv(summary_path, index=False)

training_summary

## Step 14. [디벨롭] baseline과 비교할 항목을 정리합니다

이 노트북 하나만으로 최종 결론을 내리지는 않습니다.

다음 단계에서 baseline 결과와 OpenCV 증강 결과를 같은 표로 비교하면 좋습니다.

비교할 항목은 다음과 같습니다.

- train 데이터 수가 얼마나 늘었는지
- validation loss가 baseline보다 낮아졌는지
- validation accuracy가 baseline보다 좋아졌는지
- 학습 시간이 얼마나 늘었는지
- test metric과 confusion matrix에서 실제 개선이 있었는지
- 오분류 이미지 유형이 줄었는지 또는 달라졌는지

In [ ]:
# [디벨롭] 최종 결과 분석에서 baseline과 어떤 기준으로 비교할지 미리 표로 정리합니다.
comparison_plan = pd.DataFrame([
    {"item": "데이터 수", "baseline": "원본 train만 사용", "opencv_augmented": "원본 train + OpenCV 증강 train"},
    {"item": "평가 기준", "baseline": "원본 validation/test", "opencv_augmented": "같은 원본 validation/test"},
    {"item": "모델 구조", "baseline": "SimpleCNN", "opencv_augmented": "SimpleCNN"},
    {"item": "확인할 결과", "baseline": "loss/accuracy/metric", "opencv_augmented": "loss/accuracy/metric + 시간 증가"},
])

comparison_plan

## Step 15. [디벨롭] 발표용 WHY/HOW/RESULT 문장 초안

**WHY**

실제 강아지와 고양이 사진은 조명, 방향, 흔들림, 대비가 다양하기 때문에 원본 train 데이터만으로 학습하면 변화에 약할 수 있다고 판단했습니다.

**HOW**

공정한 비교를 위해 01에서 만든 같은 split을 사용했고, validation/test는 그대로 둔 상태에서 train 이미지만 OpenCV로 증강했습니다. 모델 구조와 학습 방식은 baseline과 동일하게 유지했습니다.

**RESULT**

증강 후 train 데이터 수가 늘어났고, 같은 validation 기준에서 loss curve와 accuracy를 baseline과 비교할 수 있게 되었습니다. 이후 test metric과 오분류 이미지를 함께 확인해 실제로 일반화 성능이 좋아졌는지 판단할 예정입니다.